## AgentCore、Strandsエージェント、A2Aの入門

[A2Aプロトコル](https://a2a-protocol.org/dev/specification/)は、独立した、潜在的に不透明なAIエージェントシステム間の通信と相互運用性を促進するために設計されたオープンスタンダードです。異なるフレームワーク、言語、または異なるベンダーによって構築される可能性があるエコシステムにおいて、A2Aは共通の言語とインタラクションモデルを提供します。

[Amazon AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)は、AIエージェントやツールをデプロイして実行するための、安全でサーバーレスな専用ホスティング環境を提供します。

最近、AWSはAgentCore Runtimeの[A2Aサポート](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-a2a.html)を発表しました。

このワークショップでは、AgentCore Runtimeを使用して次のアーキテクチャを構築します：

<img src="images/architecture-getting-started.png" style="width: 80%;">

この入門ノートブックでは、2つのエージェントを構築します。最初のエージェントはAWS Docsエキスパートです。AWS Docs MCPにクエリを送信してAWSドキュメントを読み取り、検索し、推奨事項も生成します。2番目のエージェントはAWS Blogエキスパートです。Web検索を使用してAWSの最新のブログとニュースを調べます。

それでは始めましょう！

### セットアップ

依存関係をインストールします

In [7]:
%pip install -q -r requirements.txt --no-cache-dir --force-reinstall

/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


**新しいバージョンを反映させるために、環境を再起動してください！**

In [ ]:
#import IPython

# IPython.Application.instance().kernel.do_shutdown(True)

`bedrock-agentcore-starter-toolkit`のバージョンが0.1.21か確認します

In [1]:
!pip freeze | grep boto
!pip freeze | grep agentcore

OSError: [Errno 5] Input/output error

In [2]:
# ライブラリをインポート
import os
import json
import requests
import boto3
import time
from boto3.session import Session
from strands.tools import tool

# botoセッションを取得
boto_session = Session()

### 1 - 2つのエージェントのコードを作成

`agents`フォルダが存在しない場合は作成します。

In [3]:
![ ! -d "agents" ] && mkdir agents

#### 1.1 - AWS Docsエキスパートエージェント

まず、最初のエージェントコードをローカルファイルに書き込みます。このエージェントは後でAgentCore runtimeにデプロイされます。

In [3]:
%%writefile agents/strands_aws_docs.py
import os
import logging
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.multiagent.a2a import A2AServer
from strands.tools.mcp import MCPClient
from fastapi import FastAPI
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000

# MCPクライアントを作成（Managed approachで使用）
mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx",
            args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

system_prompt = """あなたはAWS Documentation MCPサーバーを搭載したAWSドキュメントアシスタントです。あなたの役割は、ユーザーがAWSドキュメントから正確で最新の情報を見つけるのを支援することです。

重要: 応答は短く、焦点を絞ったものにしてください。

ガイドライン:
- 簡潔で実用的な回答を提供する（最大3文）
- リストには箇条書きを使用する
- 冗長な説明は省略する
- MCPが利用できない場合、基本的なAWS知識を提供する
- 操作は8秒後にタイムアウト
- 完全性よりも速度を優先する

利用可能な場合は、AWSドキュメント検索ツールにアクセスできます。"""

# Managed approachでMCPClientをエージェントに渡す
# エージェントがMCPクライアントのライフサイクルを自動管理
agent = Agent(
    system_prompt=system_prompt,
    tools=[mcp_client],
    name="AWS Docsエージェント",
    description="AWS MCPを使用してAWS Docsにクエリを送信するエージェント。",
)

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

Overwriting agents/strands_aws_docs.py


#### **オプション** - ローカルテスト

このコードをローカルでテストしたい場合は、bash/ターミナルウィンドウを開いて次のスニペットを実行できます：

```bash
python agents/strands_aws_docs.py
```

サーバーがローカルで起動します。次に、別のターミナル/bashで次のコマンドを実行してテストします：

```bash
curl -X POST http://0.0.0.0:9000 \-H "Content-Type: application/json" \-d '{  "jsonrpc": "2.0",  "id": "req-001",  "method": "message/send",  "params": {  "message": {  "role": "user",  "parts": [  {  "kind": "text",  "text": "What's AWS Lambda?"  }  ],  "messageId": "d0673ab9-796d-4270-9435-451912020cd1"  }  } }' | jq .
```

MCPにクエリを送信し、AWS Lambdaを説明する回答を返します。

次のコマンドを使用して、エージェントカード情報の取得をテストすることもできます：

```bash
curl http://localhost:9000/.well-known/agent-card.json | jq .
```

#### 1.2 - AWS Blogsエキスパートエージェント

次に、2番目のエージェントコードをローカルファイルに書き込みます。

In [1]:
%%writefile agents/strands_aws_blogs_news.py
import logging
import os
import asyncio
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
import uvicorn
from fastapi import FastAPI

from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')

@tool
async def fast_internet_search(keywords: str, max_results: int = 3) -> str:
    """タイムアウト付きの高速Web検索。
    引数:
        keywords (str): 検索クエリのキーワード
        max_results (int): 最大結果数（速度のためデフォルト3）
    戻り値:
        検索結果
    """
    try:
        # より良い結果のためにAWS固有の用語を追加
        aws_keywords = f"site:aws.amazon.com {keywords} AWS"
        
        # 検索にasyncioタイムアウトを使用
        async def search_with_timeout():
            return DDGS().text(
                aws_keywords, 
                region="us-en", 
                max_results=max_results
            )
        
        results = await asyncio.wait_for(search_with_timeout(), timeout=8.0)
        
        if results:
            # 結果を簡潔にフォーマット
            formatted = []
            for i, result in enumerate(results[:max_results], 1):
                formatted.append(f"{i}. {result.get('title', 'No title')}\n   {result.get('href', '')}")
            
            return "\n".join(formatted)
        else:
            return "AWSの結果が見つかりませんでした。"
            
    except asyncio.TimeoutError:
        logger.warning(f"検索タイムアウト: {keywords}")
        return "検索がタイムアウトしました。より具体的なクエリを試してください。"
    except RatelimitException:
        logger.warning("レート制限に達しました")
        return "レート制限に達しました。しばらくしてから再試行してください。"
    except (DDGSException, Exception) as e:
        logger.error(f"検索エラー: {e}")
        return f"検索が利用できません: {str(e)[:50]}"

system_prompt = """あなたはAWS Blogエキスパートです。

重要: 応答は短く、最新のものにしてください。

ガイドライン:
- 最大3つの最新の結果を提供する
- 公式のAWSブログ投稿のみに焦点を当てる
- 簡潔な要約を使用する（結果ごとに1-2文）
- 利用可能な場合は直接リンクを含める
- 検索は8秒後にタイムアウト
- 検索が失敗した場合、制限を認める

検索戦略:
- 検索には常に「AWS」を含める
- aws.amazon.com/blogs/のコンテンツに焦点を当てる
- 最近のアナウンスを優先する"""

agent = Agent(
    system_prompt=system_prompt, 
    tools=[fast_internet_search],
    name="AWS Blog/Newsエージェント",
    description="Web上で最新のAWSブログとニュースを検索するエージェント。",
)

host, port = "0.0.0.0", 9000

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True
)

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

Writing agents/strands_aws_blogs_news.py


エージェントに必要な依存関係を含むrequirements.txtファイルを書き込みます。

In [ ]:
%%writefile agents/requirements.txt
boto3==1.40.50
bedrock-agentcore==0.1.7
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs

### 2 - AgentCore Runtimeにデプロイ

それでは、このソリューションをAgentCore Runtimeにデプロイします。

#### 2.1 - Cognito User Poolのセットアップ

エージェントをデプロイする前に、Cognito User Poolをセットアップする必要があります。これにより、エージェントにアクセスするユーザーや、Okta、Microsoft Entra IDなどの他のアイデンティティプロバイダーを検証できます。

ワークショップのいくつかのステップを簡素化するメソッドを持つヘルパークラスをインポートします。このヘルパークラスは、Cognito User Poolを作成する責任を持つメソッドをインポートします。

In [ ]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Amazon Cognitoユーザープールを設定中...")
cognito_config = (
    setup_cognito_user_pool()
)  # この出力セルからベアラートークンを取得します。
print("Cognitoの設定が完了しました ✓")

#### 2.2 - エージェント用のIAMロールを作成

##### 2.2.1 AWS Docsエージェント実行ロール

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_DOCS_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(AWS_DOCS_ROLE_NAME)

##### 2.2.2 AWS Blogsエージェント実行ロール

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_BLOG_ROLE_NAME

execution_role_arn_blogs = create_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)

##### AgentCore Runtimeでのデプロイメント用の設定を作成

次のセクションでは、[starter toolkit](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-starter-toolkit.html)を活用します。starter toolkitは、AIエージェントをAgentCore Runtimeにデプロイするために使用できるコマンドラインインターフェース（CLI）ツールキットです。

これから、AgentCore runtime内でA2Aプロトコルをサポートするエージェントを作成します。

##### 2.2.3 - 最初のエージェント（AWS Docsエージェント）を設定してデプロイします：

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_mcp_agent = Runtime()
aws_docs_agent_name="aws_docs_assistant"

region = boto_session.region_name

# デプロイメントを設定
response_aws_docs_agent = agentcore_runtime_mcp_agent.configure(
    entrypoint="agents/strands_aws_docs.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_docs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("設定が完了しました:", response_aws_docs_agent)

AgentCore Runtimeで最初のエージェントを起動します

In [ ]:
launch_result_mcp = agentcore_runtime_mcp_agent.launch()
print("起動が完了しました:", launch_result_mcp.agent_arn)

docs_agent_arn = launch_result_mcp.agent_arn

**デプロイメントステータスの確認**

デプロイメントが完了したか確認しましょう：

In [ ]:
status_response = agentcore_runtime_mcp_agent.status()
status = status_response.endpoint["status"]

print(f"最終ステータス: {status}")

##### 2.2.4 - 2番目のエージェント（AWS Blogs and Newsエージェント）を設定してデプロイします：

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_blogs = Runtime()
aws_blogs_agent_name="aws_blog_assistant"

# デプロイメントを設定
response_aws_blogs_agent = agentcore_runtime_blogs.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn_blogs,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_blogs_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A"
)

print("設定が完了しました:", response_aws_blogs_agent)

AgentCore Runtimeで2番目のエージェントを起動します

In [ ]:
launch_result_blog = agentcore_runtime_blogs.launch()
print("起動が完了しました:", launch_result_blog.agent_arn)

blog_agent_arn = launch_result_blog.agent_arn

**デプロイメントステータスの確認**

2番目のエージェントのデプロイメントが完了したか確認しましょう：

In [ ]:
status_response = agentcore_runtime_blogs.status()
status = status_response.endpoint["status"]

print(f"最終ステータス: {status}")

##### 2.2.5 - 出力をエクスポートして保存

次のノートブックで使用する変数をエクスポートします：

In [ ]:
MCP_AGENT_ID = launch_result_mcp.agent_id
MCP_AGENT_ARN = launch_result_mcp.agent_arn
MCP_AGENT_NAME = aws_docs_agent_name

BLOG_AGENT_ID = launch_result_blog.agent_id
BLOG_AGENT_ARN = launch_result_blog.agent_arn
BLOG_AGENT_NAME = aws_blogs_agent_name

COGNITO_CLIENT_ID = cognito_config.get("client_id")
COGNITO_SECRET = cognito_config.get("client_secret")
DISCOVERY_URL = cognito_config.get("discovery_url")

%store MCP_AGENT_ID
%store MCP_AGENT_ARN
%store MCP_AGENT_NAME
%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME
%store COGNITO_CLIENT_ID
%store COGNITO_SECRET
%store DISCOVERY_URL

オーケストレーターで使用できるように、エージェントのARNをSSMに保存します：

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_DOCS_AGENT_ARN, MCP_AGENT_ARN)

put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

### 3 - A2Aエージェントを呼び出す

まず、認証トークンを更新します：

In [ ]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

#### 3.1 エージェントカードの取得

最初のエージェント（AWS Docs MCPエキスパート）からエージェントカード情報を取得することから始めましょう：

In [ ]:
import logging
from uuid import uuid4
from urllib.parse import quote

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

def fetch_agent_card(agent_arn):
    # エージェントARNをURLエンコード
    escaped_agent_arn = quote(agent_arn, safe='')

    # URLを構築
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    logger.info(url)
    # 一意のセッションIDを生成
    session_id = str(uuid4())
    logger.info(f"生成されたセッションID: {session_id}")

    # ヘッダーを設定
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # リクエストを実行
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # JSONを解析して整形表示
        agent_card = response.json()
        logger.info(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        logger.error(f"エージェントカードの取得エラー: {e}")
        return None

In [ ]:
fetch_agent_card(docs_agent_arn)

次に、2番目のエージェント（AWS Blogs and Newsエキスパート）のエージェントカードを確認しましょう： 

In [ ]:
fetch_agent_card(blog_agent_arn)

#### 3.2 - エージェントをテスト

それでは、A2Aを使用して最初のエージェントを呼び出しましょう：

In [ ]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # リクエストタイムアウトを5分に設定

def format_agent_response(response):
    """人間が読みやすいようにエージェントの応答を抽出してフォーマットする。"""
    # アーティファクトからメインの応答テキストを取得
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text
    
    # フォールバック: 履歴からすべてのエージェントメッセージを連結
    agent_messages = [
        msg.parts[0].root.text 
        for msg in response.history 
        if msg.role.value == 'agent' and msg.parts
    ]
    return ''.join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(agent_arn, message: str):
    # 環境変数からランタイムURLを取得
    escaped_agent_arn = quote(agent_arn, safe='')

    # URLを構築
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # 一意のセッションIDを生成
    session_id = str(uuid4())
    print(f"生成されたセッションID: {session_id}")

    # AgentCore用の認証ヘッダーを追加
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # ランタイムURLからエージェントカードを取得
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # エージェントカードには正しいURLが含まれています（この場合はruntime_urlと同じ）
        # 手動でのオーバーライドは不要 - これはパスベースのマウントパターンです

        # ファクトリーを使用してクライアントを作成
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # 同期応答には非ストリーミングモードを使用
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # メッセージを作成して送信
        msg = create_message(text=message)

        # streaming=Falseの場合、正確に1つの結果が生成されます
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) タプル
                task, update_event = event
                logger.info(f"タスク: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"更新: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # その他の応答タイプのフォールバック
                logger.info(f"応答: {str(event)}")
                return event

In [ ]:
result = await send_sync_message(docs_agent_arn, "what is DynamoDB")
formatted_output = format_agent_response(result)
print(formatted_output)

次に、2番目のエージェントをテストしましょう：

In [ ]:
result = await send_sync_message(blog_agent_arn, "Give me the latest published blog for Bedrock AgentCore?")
formatted_output = format_agent_response(result)
print(formatted_output)

以下は、エージェントが実行したステップを示すより詳細な出力です。

エージェントに尋ねる質問を変更して、ステップバイステップの結果を確認してください。

In [ ]:
def format_agent_trace(response):
    """エージェントの応答を読みやすい呼び出しトレースとしてフォーマットする。"""
    print("=" * 60)
    print("🔍 エージェント実行トレース")
    print("=" * 60)
    
    # コンテキスト情報
    print(f"📋 コンテキストID: {response.context_id}")
    print(f"🆔 タスクID: {response.id}")
    print(f"📊 ステータス: {response.status.state.value}")
    print(f"⏰ 完了時刻: {response.status.timestamp}")
    print()
    
    # 履歴をトレース
    print("🔄 実行フロー:")
    print("-" * 40)
    
    for i, msg in enumerate(response.history, 1):
        role_icon = "👤" if msg.role.value == "user" else "🤖"
        text = msg.parts[0].root.text if msg.parts else "[コンテンツなし]"
        
        # トレース表示用に長いメッセージを切り詰め
        if len(text) > 80:
            text = text[:77] + "..."
            
        print(f"{i:2d}. {role_icon} {msg.role.value.upper()}: {text}")
    
    print()
    print("✅ 最終結果:")
    print("-" * 40)
    
    # 最終アーティファクト
    if response.artifacts:
        final_text = response.artifacts[0].parts[0].root.text
        print(final_text[:200] + "..." if len(final_text) > 200 else final_text)
    
    print("=" * 60)

In [ ]:
format_agent_trace(result)

おめでとうございます！Amazon AgentCore RuntimeでA2Aプロトコルを使用して最初のエージェントをデプロイしました！

それでは、次のラボに進みましょう。